In [ ]:
import dynamiqs as dq
dq.set_precision('double')
import numpy as np
from scipy.sparse.linalg import eigs
import scipy.sparse as sp
import scipy.sparse.linalg as spla

import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
kappa_b = 5 / 10.4*2
E_J = 37*2*np.pi /2
phi_a = 0.11
phi_b = 0.204
epsilon_p = 0.02
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2

In [ ]:
n_a = 25

# --- operators ---
a = dq.destroy(n_a)
Ia = dq.eye(n_a)

# pure dissipative cat stabilization
H = 0 * Ia
jump_op_1 = np.sqrt(kappa_1)*a

# --- convert to numpy ---
I_np = np.eye(n_a, dtype=np.complex128)
L_np_1 = jump_op_1.to_numpy()
H_np = H.to_numpy()

LdL_1 = L_np_1.conj().T @ L_np_1

In [ ]:
alpha_sq_list = np.linspace(1,2.5,10)
bit_flip_rate_list = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    jump_op_2 = np.sqrt(kappa_2) * (a @ a - alpha_sq * Ia)

    # --- convert to numpy ---
    L_np_2 = jump_op_2.to_numpy()
    LdL_2 = L_np_2.conj().T @ L_np_2

    # --- Liouvillian in vectorized form (column stacking) ---
    liouvillian = (
        -1j * (np.kron(I_np, H_np) - np.kron(H_np.T, I_np))
        + np.kron(L_np_2.conj(), L_np_2)
        - 0.5 * np.kron(I_np, LdL_2)
        - 0.5 * np.kron(LdL_2.T, I_np)
        + np.kron(L_np_1.conj(), L_np_1)
        - 0.5 * np.kron(I_np, LdL_1)
        - 0.5 * np.kron(LdL_1.T, I_np)
    )

    vals = np.linalg.eigvals(liouvillian).real
    bit_flip_rate_list.append(-np.sort(vals)[-2])


In [ ]:
bit_flip_rate_list_arnoldi = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    jump_op_2 = np.sqrt(kappa_2) * (a @ a - alpha_sq * Ia)

    # --- convert to numpy ---
    L_np_2 = jump_op_2.to_numpy()
    LdL_2 = L_np_2.conj().T @ L_np_2

    # --- Liouvillian in vectorized form (column stacking) ---
    liouvillian = (
        -1j * (np.kron(I_np, H_np) - np.kron(H_np.T, I_np))
        + np.kron(L_np_2.conj(), L_np_2)
        - 0.5 * np.kron(I_np, LdL_2)
        - 0.5 * np.kron(LdL_2.T, I_np)
        + np.kron(L_np_1.conj(), L_np_1)
        - 0.5 * np.kron(I_np, LdL_1)
        - 0.5 * np.kron(LdL_1.T, I_np)
    )

    vals, _ = eigs(liouvillian, k=2, which = "SM")
    vals = vals.real
    bit_flip_rate_list_arnoldi.append(-np.sort(vals)[-2].real)


In [ ]:
pert_th = kappa_1*alpha_sq_list*np.exp(-4*alpha_sq_list)+kappa_1**2/(2*kappa_2)*np.exp(-2*alpha_sq_list)

In [ ]:
n_a = 20
n_b = 11
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

# Correct nonlinear term
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)
drive_op = b_tot + b_tot.dag()

La = np.sqrt(kappa_1) * a_tot
Lb = np.sqrt(kappa_b) * b_tot

# Convert to sparse CSC
La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
I = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )

In [ ]:
rates_list_arnoldi = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    epsilon_d = 2*alpha_sq*g
    H = g * non_linear_op + (epsilon_d / 2) * drive_op
    H = sp.csc_matrix(H.to_numpy().astype(np.complex128))
    
    Lio = (
    -1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
    + dissipator(La)
    + dissipator(Lb)
    )

    vals, vecs = spla.eigs(Lio, k=6, sigma=1e-12, which="LM", tol=1e-10)
    rates_list_arnoldi.append(-np.sort(vals)[-2].real)

In [ ]:
g/2/np.pi*1e3/2

In [ ]:
import numpy as np
w_b = 25.338776456203686*2
T = 2*np.pi/2/w_b
import json
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/arnoldi_output/arnoldi_re_22_04_2026.json", "r") as f:
    phi_j = np.array(json.load(f)).T
lam_j = -np.log(phi_j)/T
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/arnoldi_output/arnoldi_re_22_04_2026_bis.json", "r") as f:
    phi_j_bis = np.array(json.load(f)).T
lam_j_bis = -np.log(phi_j_bis)/T

with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/arnoldi_output/arnoldi_re_22_04_2026_kappa_b_3.json", "r") as f:
    phi_j_ter = np.array(json.load(f)).T
lam_j_ter = -np.log(phi_j_ter)/T
lam_j_ter

lam_j_4 = -np.log(np.array([0.9999999998472493, 0.9999999999653225]))/T
lam_j_4

with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/arnoldi_output/arnoldi_re_22_04_2026_full_ATS.json", "r") as f:
    phi_j_full = np.array(json.load(f)).T
lam_j_full = -np.log(phi_j_full)/T
lam_j_full

with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/arnoldi_output/arnoldi_re_22_04_2026_full_ATS_larger_kappa_1.json", "r") as f:
    phi_j_full_larger_kappa_1 = np.array(json.load(f)).T
phi_j_full_larger_kappa_1 = -np.log(phi_j_full_larger_kappa_1)/T
phi_j_full_larger_kappa_1

In [ ]:
[0.999999998473836, 0.9999999998681497, 0.9999999999886534]
[0.999999997429295, 0.9999999993549195, 0.9999999625524286]

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
bit_flip_rate_arr = np.array(bit_flip_rate_list)
plt.scatter(alpha_sq_list, bit_flip_rate_arr*1e9, label="Exact diag. one-mode")
plt.scatter(np.linspace(1,2.5,10), lam_j_bis*1e9, label="""Arnoldi two-mode time-dependent drive \n quite fast (2'40" per point)""")
#plt.scatter(np.linspace(1,2.5,5), lam_j_ter*1e9, label="""Arnoldi two-mode time-dependent drive \n"""+r"$\kappa_b=0.6$"+" GHz")
#plt.scatter(np.linspace(2,2.5,2), lam_j_4*1e9, label="""Arnoldi two-mode time-dependent drive \n"""+r"$\kappa_b=0.4$"+" GHz")
plt.scatter(np.linspace(1,2.5,5), lam_j_full*1e9, label="""Arnoldi two-mode full ATS \n"""+r"$\kappa_b=1$"+" GHz")
#plt.scatter(np.linspace(1,2.5,5), phi_j_full_larger_kappa_1*1e9, label="""Arnoldi two-mode full ATS \n"""+r"$\kappa_b=1$"+" GHz")
plt.yscale("log")
plt.xlabel(r"$\alpha^2$")
plt.ylabel("Bit flip rate (Hz)")
plt.title(r"$g_2 = $"+str(round(g/2/np.pi*1e3/2, 1))+" MHz")
plt.legend(bbox_to_anchor=(1, 2))
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
bit_flip_rate_arr = np.array(bit_flip_rate_list)
bit_flip_rate_arr_arnoldi = np.array(bit_flip_rate_list_arnoldi)
rates_arr_arnoldi = np.array(rates_list_arnoldi)


plt.figure()
plt.plot(alpha_sq_list, bit_flip_rate_arr*1e9, label="Exact diag. one-mode", ls="-")
plt.plot(alpha_sq_list, bit_flip_rate_arr_arnoldi*1e9, label="Arnoldi default param. one-mode", ls="--")
plt.plot(alpha_sq_list, pert_th*1e9, label="Perturbative res. kerr cat")
plt.plot(alpha_sq_list, rates_arr_arnoldi*1e9, label="Arnoldi two-mode", ls=":")
plt.yscale("log")
plt.xlabel(r"$\alpha^2$")
plt.ylabel("Bit flip rate (Hz)")
plt.title(r"$g_2 = $"+str(round(g/2/np.pi*1e3/2, 1))+" MHz")
plt.legend()
plt.grid(True)
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/switching_rate/comparison_arnoldi_two_modes_one_mode_17_04_2026.pdf")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
bit_flip_rate_arr = np.array(bit_flip_rate_list)
bit_flip_rate_arr_arnoldi = np.array(bit_flip_rate_list_arnoldi)
rates_arr_arnoldi = np.array(rates_list_arnoldi)


plt.figure()
plt.plot(alpha_sq_list, bit_flip_rate_arr*1e9, label="Exact diag. one-mode", ls="-")
plt.plot(alpha_sq_list, bit_flip_rate_arr_arnoldi*1e9, label="Arnoldi default param. one-mode", ls="--")
plt.plot(alpha_sq_list, pert_th*1e3*1.5, label="Perturbative res. kerr cat")
plt.plot(alpha_sq_list, rates_arr_arnoldi*1e9, label="Arnoldi two-mode", ls=":")
plt.yscale("log")
plt.xlabel(r"$\alpha^2$")
plt.ylabel("Bit flip rate (Hz)")
plt.title(r"$g_2 = $"+str(round(g/2/np.pi*1e3/2, 1))+" MHz")
plt.legend()
plt.grid(True)
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/switching_rate/comparison_arnoldi_two_modes_one_mode_17_04_2026.pdf")
plt.show()

In [ ]:
rates_test = []
alpha_sq_list = np.linspace(1,4,5)
for kappa_b in np.linspace(0.01, 0.05, 3):
    # --- constants ---
    ħ = 1
    e = 1
    phi_quantum = 1 / 2

    w_a = 25.338776456203686
    E_J = 37*2*np.pi /2
    phi_a = 0.11
    phi_b = 0.204
    epsilon_p = 0.02
    g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
    kappa_2 = 4*g**2/kappa_b
    kappa_1 = 1e-6
    n_a = 25
    n_b = 13
    N = n_a * n_b

    # Operators
    a = dq.destroy(n_a)
    b = dq.destroy(n_b)
    Ia = dq.eye(n_a)
    Ib = dq.eye(n_b)

    a_tot = dq.tensor(a, Ib)
    b_tot = dq.tensor(Ia, b)

    # Correct nonlinear term
    term = dq.tensor(a @ a, dq.dag(b))
    non_linear_op = term + dq.dag(term)
    drive_op = b_tot + b_tot.dag()

    La = np.sqrt(kappa_1) * dq.dag(a_tot)
    Lb = np.sqrt(kappa_b) * b_tot

    # Convert to sparse CSC
    La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
    Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
    I = sp.identity(N, dtype=np.complex128, format="csc")

    def dissipator(L):
        LdL = L.getH() @ L
        return (
            sp.kron(L.conjugate(), L, format="csc")
            - 0.5 * sp.kron(I, LdL, format="csc")
            - 0.5 * sp.kron(LdL.T, I, format="csc")
        )
    rates_list_arnoldi = []
    for alpha_sq in alpha_sq_list:
        print(alpha_sq)
        epsilon_d = 2*alpha_sq*g
        H = g * non_linear_op + (epsilon_d / 2) * drive_op
        H = sp.csc_matrix(H.to_numpy().astype(np.complex128))
        
        Lio = (
        -1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
        + dissipator(La)
        + dissipator(Lb)
        )

        vals, vecs = spla.eigs(Lio, k=6, sigma=1e-12, which="LM", tol=1e-10)
        rates_list_arnoldi.append(-np.sort(vals)[-2].real)
    rates_test.append(rates_list_arnoldi)

In [ ]:
pert_th =np.exp(-1.5*alpha_sq_list)# 
import matplotlib.pyplot as plt
plt.figure()
plt.plot(alpha_sq_list, pert_th*1e3*3, label="", ls="--")
for (i,kappa_b) in enumerate(np.linspace(0.02, 0.1, 3)):
    print(kappa_b/4/np.sqrt(2)/g/np.sqrt(alpha_sq_list[0]))
    rates_arr_arnoldi = np.array(rates_test[i])
    plt.plot(alpha_sq_list, rates_arr_arnoldi*1e9, label=r"$\kappa_b = $"+str(round(kappa_b,2))+" GHz")
plt.yscale("log")
plt.xlabel(r"$\alpha^2$")
plt.ylabel("Bit flip rate (Hz)")
plt.title(r"$g_2 = $"+str(round(g/2/np.pi*1e3/2, 1))+" MHz")
plt.legend()
plt.grid(True)
plt.show()
